# UD1 — Matemáticas que ya usamos en el pipeline

**Sesión guiada: 60 minutos.** En cada parada ejecutamos un ejemplo pequeño del caso CadenaRetail y lo conectamos con una decisión de datos. No es una clase de demostraciones: queremos reconocer las ideas matemáticas que ya aparecen en los sistemas que construiremos.

> La estadística se trabaja explícitamente en su propio bloque. Este notebook recorre la matemática discreta, las relaciones, la lógica y la complejidad de la cápsula normativa.

### Dónde encontrarlo

Este notebook está en `ud01-introduccion-big-data/02-ejemplos/` y está enlazado desde el [mapa de matemáticas en el módulo](../01-teoria/00-estadistica-y-base-matematica/UD1_P0_02_Donde_Aparecen_las_Matematicas.md).

Material relacionado:

- [Cápsula matemática normativa](../01-teoria/00-estadistica-y-base-matematica/UD1_P0_Capsula_Matematica_Normativa.md)
- [Actividad formativa de CadenaRetail](../03-practicas/fundamentos-matematicos-cadenaretail/UD1_P0_Actividad_Fundamentos_Matematicos_CadenaRetail.md)
- [Estadística para Big Data](../01-teoria/00-estadistica-y-base-matematica/UD1_P0_Estadistica_para_BigData.md)

### Ruta y tiempos orientativos

| Minutos | Idea | Decisión que ilustra |
|---:|---|---|
| 0–4 | Presentación del caso y objetivo | La matemática ayuda a justificar decisiones del pipeline. |
| 4–12 | Conjuntos y pertenencia | ¿El SKU vendido existe en el catálogo? |
| 12–22 | Relaciones, claves y joins | ¿Unir las tablas conserva el número correcto de ventas? |
| 22–32 | Lógica y reglas | ¿Qué filas se publican, revisan o aíslan? |
| 32–40 | Grafos y dependencias | ¿Qué tareas deben terminar antes de curar y publicar? |
| 40–48 | Conteos y combinaciones | ¿Cuántos valores, grupos o combinaciones podríamos producir? |
| 48–57 | Complejidad | ¿Qué cambia al pasar de cinco ventas a millones? |
| 57–60 | Síntesis | ¿Qué concepto matemático hay detrás de una decisión técnica? |

Ejecuta las celdas en orden y detente en las preguntas. Los enlaces de cada sección señalan dónde volverás a encontrar la idea durante el módulo.

## 0. El caso: ventas, catálogo y carga

La cadena recibe líneas de venta de varios canales. El catálogo maestro indica qué SKU son oficiales. Usaremos cinco líneas simplificadas de la actividad de CadenaRetail.

In [1]:
import pandas as pd
from itertools import product
from collections import deque
from IPython.display import display

catalogo = {'TS-ROJO-M', 'TS-AZUL-M', 'TS-NEGRO-L', 'PANT-VAQ-42', 'ZAP-BLANCO-39'}
ventas = pd.DataFrame([
    {'pedido_id': 1001, 'sku': 'TS-ROJO-M',     'unidades': 2, 'precio_unitario': 12.50, 'precio_total': 25.00, 'dias_carga': 1},
    {'pedido_id': 1002, 'sku': 'TSHIRT_4421',   'unidades': 1, 'precio_unitario': 12.50, 'precio_total': 12.50, 'dias_carga': 1},
    {'pedido_id': 1003, 'sku': 'TS-AZUL-M',     'unidades': 3, 'precio_unitario': 10.00, 'precio_total': 25.00, 'dias_carga': 1},
    {'pedido_id': 1004, 'sku': 'GORRA-VERDE-U', 'unidades': 1, 'precio_unitario':  8.00, 'precio_total':  8.00, 'dias_carga': 3},
    {'pedido_id': 1005, 'sku': 'PANT-VAQ-42',   'unidades': 1, 'precio_unitario': 30.00, 'precio_total': 30.00, 'dias_carga': 1},
])

display(ventas)

,pedido_id,sku,unidades,precio_unitario,precio_total,dias_carga
0,1001,TS-ROJO-M,2,12.5,25.0,1
1,1002,TSHIRT_4421,1,12.5,12.5,1
2,1003,TS-AZUL-M,3,10.0,25.0,1
3,1004,GORRA-VERDE-U,1,8.0,8.0,3
4,1005,PANT-VAQ-42,1,30.0,30.0,1


## 1. Conjuntos: comprobar pertenencia (4–12 min)

Un conjunto permite razonar sobre pertenencia, coincidencias y diferencias. Aquí comparamos los SKU que aparecen en ventas con los SKU oficiales del catálogo.

**Pregunta antes de ejecutar:** ¿qué códigos no se podrán asociar a un producto reconocido?

**Dónde reaparece:** la actividad formativa de [fundamentos matemáticos de CadenaRetail](../03-practicas/fundamentos-matematicos-cadenaretail/UD1_P0_Actividad_Fundamentos_Matematicos_CadenaRetail.md) practica esta misma decisión; en UD2 aparece al integrar fuentes y validar dominios en [integración y calidad incremental](../../ud02-almacenamiento-ingesta/01-teoria/UD2_03_Integracion_y_calidad_incremental.md).

In [2]:
vendidos = set(ventas['sku'])
print('En ventas y en catálogo:', vendidos & catalogo)
print('En ventas, pero no en catálogo:', vendidos - catalogo)
print('En catálogo, pero no en estas ventas:', catalogo - vendidos)

# Los conjuntos no conservan duplicados: revisamos filas con la tabla.
print('Líneas de venta:', len(ventas))
print('SKU distintos en ventas:', ventas['sku'].nunique())

En ventas y en catálogo: {'TS-AZUL-M', 'TS-ROJO-M', 'PANT-VAQ-42'}
En ventas, pero no en catálogo: {'TSHIRT_4421', 'GORRA-VERDE-U'}
En catálogo, pero no en estas ventas: {'ZAP-BLANCO-39', 'TS-NEGRO-L'}
Líneas de venta: 5
SKU distintos en ventas: 5


**Idea clave:** un `set` responde si algo está o no está; no reemplaza a la tabla. La tabla conserva cuántas veces aparece un SKU y a qué pedido pertenece.

## 2. Relaciones, claves y joins (12–22 min)

Una relación conecta entidades: una venta referencia un producto por su SKU. Un `join` usa esa clave para añadir información del catálogo. Si el catálogo tiene varias filas para la misma clave, una venta puede convertirse en varias filas y las métricas quedar infladas.

**Dónde reaparece:** los joins y upserts se usan en [UD2](../../ud02-almacenamiento-ingesta/01-teoria/UD2_03_Integracion_y_calidad_incremental.md); la misma idea de claves y relaciones sostiene el [modelado mínimo para BI en UD4](../../ud04-bi-orquestacion/01-teoria/Parte-I_BI/UD4_02_Modelado_minimo_para_BI.md).

In [3]:
catalogo_df = pd.DataFrame([
    {'sku': sku, 'producto': producto}
    for sku, producto in [
        ('TS-ROJO-M', 'Camiseta roja M'),
        ('TS-AZUL-M', 'Camiseta azul M'),
        ('TS-NEGRO-L', 'Camiseta negra L'),
        ('PANT-VAQ-42', 'Pantalón vaquero 42'),
        ('ZAP-BLANCO-39', 'Zapato blanco 39'),
    ]
])
ventas_catalogadas = ventas.merge(
    catalogo_df,
    on='sku',
    how='left',
    validate='many_to_one',
    indicator=True,
)
display(ventas_catalogadas[['pedido_id', 'sku', 'producto', '_merge']])
print('Filas antes del join:', len(ventas), '| después:', len(ventas_catalogadas))
assert len(ventas_catalogadas) == len(ventas)

,pedido_id,sku,producto,_merge
0,1001,TS-ROJO-M,Camiseta roja M,both
1,1002,TSHIRT_4421,NaN,left_only
2,1003,TS-AZUL-M,Camiseta azul M,both
3,1004,GORRA-VERDE-U,NaN,left_only
4,1005,PANT-VAQ-42,Pantalón vaquero 42,both


Filas antes del join: 5 | después: 5


El argumento `validate='many_to_one'` expresa una regla sobre la relación: puede haber muchas ventas del mismo SKU, pero una sola fila de catálogo por SKU. Si esa regla no se cumple, interesa enterarse antes de sumar ventas.

## 3. Lógica: convertir decisiones en reglas (22–32 min)

La lógica algorítmica aparece cada vez que escribimos condiciones. Cada fila puede pasar o fallar reglas independientes; no ocultamos los registros que fallan, los marcamos para decidir qué hacer.

**Dónde reaparece:** las reglas de validez, rango y consistencia se trabajan en la [guía EDA y calidad de UD1](../01-teoria/04-eda-calidad-duckdb/UD1_02_EDA_y_calidad.md) y vuelven a ejecutarse en las transformaciones de UD2.

In [4]:
ventas['sku_valido'] = ventas['sku'].isin(catalogo)
ventas['importe_valido'] = (
    (ventas['precio_total'] - ventas['unidades'] * ventas['precio_unitario']).abs() <= 0.01
)
ventas['sla_carga_ok'] = ventas['dias_carga'] <= 1

ventas['requiere_revision'] = ~(
    ventas['sku_valido'] & ventas['importe_valido'] & ventas['sla_carga_ok']
)
display(ventas[['pedido_id', 'sku_valido', 'importe_valido', 'sla_carga_ok', 'requiere_revision']])

,pedido_id,sku_valido,importe_valido,sla_carga_ok,requiere_revision
0,1001,True,True,True,False
1,1002,False,True,True,True
2,1003,True,False,True,True
3,1004,False,True,False,True
4,1005,True,True,True,False


Lee las condiciones en castellano:

- `sku` pertenece al catálogo.
- `precio_total` coincide con unidades × precio unitario, con tolerancia de un céntimo.
- la carga llegó en el plazo provisional de un día.
- solo si las tres condiciones se cumplen, la fila pasa sin revisión.

La regla marca casos; la decisión (corregir, poner en cuarentena o aceptar) requiere contexto de negocio.

## 4. Grafos: dependencias como en Airflow (32–40 min)

Un grafo tiene nodos y conexiones. En un pipeline, los nodos pueden ser tareas y las flechas indican dependencias. Airflow representa estos flujos mediante DAG (*Directed Acyclic Graph*): grafo dirigido sin ciclos.

Este ejemplo no ejecuta Airflow ni instala su paquete; hace visible la estructura que estudiarás en [Orquestación con Airflow, UD4](../../ud04-bi-orquestacion/01-teoria/ParteII-Orquestacion/UD4_07_Orquestacion_con_Airflow.md).

In [5]:
tareas = ['tienda_raw', 'web_raw', 'validar_tienda', 'validar_web', 'curar', 'publicar']
dependencias = [
    ('tienda_raw', 'validar_tienda'),
    ('web_raw', 'validar_web'),
    ('validar_tienda', 'curar'),
    ('validar_web', 'curar'),
    ('curar', 'publicar'),
]

# Orden topológico sencillo: una tarea entra cuando todos sus requisitos terminaron.
previas = {tarea: set() for tarea in tareas}
siguientes = {tarea: set() for tarea in tareas}
for origen, destino in dependencias:
    previas[destino].add(origen)
    siguientes[origen].add(destino)

listos = deque(tarea for tarea in tareas if not previas[tarea])
orden = []
while listos:
    tarea = listos.popleft()
    orden.append(tarea)
    for destino in siguientes[tarea]:
        previas[destino].remove(tarea)
        if not previas[destino]:
            listos.append(destino)

if len(orden) != len(tareas):
    raise ValueError('Hay un ciclo: no existe un orden válido de ejecución.')
print('Dependencias:', dependencias)
print('Un orden posible:', ' → '.join(orden))

Dependencias: [('tienda_raw', 'validar_tienda'), ('web_raw', 'validar_web'), ('validar_tienda', 'curar'), ('validar_web', 'curar'), ('curar', 'publicar')]
Un orden posible: tienda_raw → web_raw → validar_tienda → validar_web → curar → publicar


**Pregunta:** ¿podrían ejecutarse las dos validaciones en paralelo? Sí: ambas dependen de su propia llegada raw, no una de la otra. `curar` espera a las dos; `publicar` espera a `curar`. Leer el DAG ayuda a razonar sobre el orden sin memorizar una figura.

## 5. Conteos, cardinalidad y combinaciones (40–48 min)

Contar y comparar cardinalidades también es razonamiento discreto. En sistemas de datos contamos filas, claves distintas y grupos. Además, al cruzar categorías podemos crear combinaciones posibles que no necesariamente aparecen en los datos observados.

**Dónde reaparece:** métricas de unicidad y recuentos aparecen en EDA de UD1; agregaciones, joins y consultas se repiten en UD2 y en el [benchmark de motores de UD3](../../ud03-procesamiento-distribuido/03-practicas/Spark_Labs/UD3_Lab_Benchmark_Pandas_DuckDB_Spark.md).

In [6]:
print('Filas de venta:', len(ventas))
print('Pedidos distintos:', ventas['pedido_id'].nunique())
print('SKU distintos:', ventas['sku'].nunique())
print('Líneas por SKU:')
display(ventas.groupby('sku').size().rename('numero_lineas').to_frame())

fechas = ['lunes', 'martes', 'miércoles', 'jueves', 'viernes']
canales = ['tienda', 'web', 'app']
productos = ['camiseta', 'pantalón', 'zapato', 'gorra']
combinaciones_posibles = list(product(fechas, canales, productos))
print(f'{len(fechas)} fechas × {len(canales)} canales × {len(productos)} tipos = {len(combinaciones_posibles)} grupos posibles')
print('Ejemplos:', combinaciones_posibles[:5])

Filas de venta: 5
Pedidos distintos: 5
SKU distintos: 5
Líneas por SKU:


,numero_lineas
sku,
GORRA-VERDE-U,1
PANT-VAQ-42,1
TS-AZUL-M,1
TS-ROJO-M,1
TSHIRT_4421,1


5 fechas × 3 canales × 4 tipos = 60 grupos posibles
Ejemplos: [('lunes', 'tienda', 'camiseta'), ('lunes', 'tienda', 'pantalón'), ('lunes', 'tienda', 'zapato'), ('lunes', 'tienda', 'gorra'), ('lunes', 'web', 'camiseta')]


Las 60 combinaciones son un máximo teórico de grupos en este ejemplo; no significan que los 60 hayan ocurrido. La misma cautela vale para `group by`, particiones y cruces: distinguir posibilidades de observaciones evita presentar como real algo que solo era posible.

## 6. Complejidad: el tamaño cambia la decisión (48–57 min)

La complejidad estudia cómo crece el trabajo al crecer los datos. No hace falta medir segundos para detectar una diferencia de escala: basta contar comparaciones.

**Dónde reaparece:** la actividad matemática de UD1 compara una búsqueda por recorrido con la pertenencia a un conjunto; en UD3 se mide el coste real de pandas, DuckDB y Spark con un [benchmark reproducible](../../ud03-procesamiento-distribuido/03-practicas/Spark_Labs/UD3_Lab_Benchmark_Pandas_DuckDB_Spark.md).

In [7]:
# Búsqueda ingenua: cada venta recorre el catálogo hasta encontrar coincidencia.
def comparaciones_ingenuas(numero_ventas, numero_sku):
    return numero_ventas * numero_sku

ventas_pequenas, catalogo_pequeno = 10, 5
ventas_grandes, catalogo_grande = 10_000_000, 200_000

print('Ejemplo pequeño, máximo de comparaciones:', comparaciones_ingenuas(ventas_pequenas, catalogo_pequeno))
print('Escenario grande, máximo de comparaciones:', comparaciones_ingenuas(ventas_grandes, catalogo_grande))
print('Con un conjunto hash: se construye el índice del catálogo y luego se hace una comprobación media por venta.')

Ejemplo pequeño, máximo de comparaciones: 50
Escenario grande, máximo de comparaciones: 2000000000000
Con un conjunto hash: se construye el índice del catálogo y luego se hace una comprobación media por venta.


El recorrido ingenuo puede necesitar hasta \(n \times m\) comparaciones para \(n\) ventas y \(m\) productos. Un conjunto hash suele permitir pertenencia en tiempo medio constante, a cambio de construir y mantener el conjunto y usar memoria. **No es una promesa universal de tiempo constante ni una medición de rendimiento**: para sistemas grandes hay que medir y considerar sus datos y herramientas.

## 7. Cierre (57–60 min)

Completa oralmente o en una nota:

1. Comparar SKU con el catálogo usa la idea de…
2. Un join puede inflar las ventas cuando…
3. Una regla `SI ... ENTONCES ...` es lógica aplicada a…
4. Un DAG de Airflow representa…
5. Comparar \(n \times m\) con una búsqueda por conjunto sirve para razonar sobre…

### Mapa de reaparición

| Hoy | Lo volverás a encontrar |
|---|---|
| Conjuntos, reglas y búsqueda por SKU | [Actividad matemática de UD1](../03-practicas/fundamentos-matematicos-cadenaretail/UD1_P0_Actividad_Fundamentos_Matematicos_CadenaRetail.md) |
| Calidad y condiciones | [EDA y calidad, UD1](../01-teoria/04-eda-calidad-duckdb/UD1_02_EDA_y_calidad.md) |
| Claves, relaciones, joins y cardinalidad | [Integración de UD2](../../ud02-almacenamiento-ingesta/01-teoria/UD2_03_Integracion_y_calidad_incremental.md) |
| Crecimiento de coste | [Benchmark pandas/DuckDB/Spark, UD3](../../ud03-procesamiento-distribuido/03-practicas/Spark_Labs/UD3_Lab_Benchmark_Pandas_DuckDB_Spark.md) |
| Grafos de dependencias | [Airflow y DAGs, UD4](../../ud04-bi-orquestacion/01-teoria/ParteII-Orquestacion/UD4_07_Orquestacion_con_Airflow.md) |

**Idea final:** no vamos a estudiar formalmente todas estas ramas. Sí vamos a reconocerlas y usarlas para justificar cómo integramos, validamos, ordenamos y escalamos un pipeline.